# 📈 Attrition Risk Results Preview
**Objective:** Since localhost ports are blocked in this environment, this notebook replicates the Streamlit dashboard visualizations.

### This notebook will:
1. Load the trained XGBoost model and master data.
2. Generate risk scores for **2026** employees.
3. Show the Top Drivers of attrition.
4. List the **Saveable Stars** (High Performer + High Risk).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import sys
from IPython.display import display, HTML

# Add project root to path for imports
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import PATH_OUTPUT, PATH_MODEL_OUTPUT, BASE_FEATURES, CATEGORICAL_FEATURES, TARGET_COLUMN
from feature_engine import prepare_model_data

plt.style.use('ggplot')
print("✅ Libraries loaded.")

## 1. Load Data & Trained Model

In [ ]:
# Load Model
model_path = os.path.join(PATH_MODEL_OUTPUT, 'xgboost_model.joblib')
model = joblib.load(model_path)

# Load Data
df = pd.read_parquet(os.path.join(PATH_OUTPUT, 'Model_Ready_Data.parquet'))

print(f"✅ Model loaded from: {model_path}")
print(f"✅ Data loaded: {len(df):,} rows")

## 2. Generate 2026 Risk Profile
We will now take the latest snapshot for every employee in **2026** and calculate their risk score using the model.

In [ ]:
# Prepare 2026 data
X_train, y_train, X_test, y_test, test_df, feat_names, _ = prepare_model_data(df, prediction_year=2026)

# Ensure feature alignment
try:
    # Match features used during training
    X_test_aligned = X_test[model.get_booster().feature_names]
except:
    X_test_aligned = X_test

# Predict Risk
y_prob = model.predict_proba(X_test_aligned)[:, 1]
test_df['Risk_Score'] = y_prob
test_df['Risk_Category'] = pd.cut(y_prob, bins=[0, 0.4, 0.7, 1.0], labels=['Low', 'Medium', 'High'])

print(f"✅ Risk scores generated for {len(test_df):,} active employees.")
test_df[['Employee_ID', 'SNAPSHOT_DATE', 'Risk_Score', 'Risk_Category']].head()

## 3. Visualizations

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(test_df['Risk_Score'], bins=50, kde=True, color='teal')
plt.axvline(0.5, color='red', linestyle='--', label='Default Threshold (0.5)')
plt.title('2026 Risk Score Distribution')
plt.xlabel('Probability of Attrition (6-month Window)')
plt.ylabel('Employee Count')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Basic Feature Importance from XGBoost
imps = pd.DataFrame({
    'Feature': feat_names,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=imps, palette='viridis')
plt.title('Top 15 Drivers of Attrition (XGBoost Feature Importance)')
plt.show()

## 4. Priority Action Lists

In [ ]:
print("⭐ SAVEABLE STARS (High Performance & High Risk)")
stars = test_df[
    (test_df['Risk_Score'] > 0.6) & 
    (test_df['Performance_Score'].isin([4, 5]))
].sort_values('Risk_Score', ascending=False)

cols_to_show = ['Employee_ID', 'Risk_Score', 'Performance_Score', 'Management_Level_Agg', 'Total_Tenure_Years']
cols_to_show = [c for c in cols_to_show if c in stars.columns]

if not stars.empty:
    display(stars[cols_to_show].head(20))
else:
    print("No high-risk stars found in current snapshot.")

In [ ]:
print("🔴 FULL HIGH RISK LIST (Risk > 0.7)")
high_risk = test_df[test_df['Risk_Score'] > 0.7].sort_values('Risk_Score', ascending=False)

if not high_risk.empty:
    display(high_risk[cols_to_show].head(20))
else:
    print("No employees flagged with risk > 0.7.")